# S&P 500 Stock Price Analysis (2014–2017)

Backward-looking analysis of daily prices for ~505 S&P 500 constituents over 1,007 trading days. The pipeline loads the raw CSV into a local PostgreSQL database, applies split adjustment, then runs a series of SQL queries covering returns, volatility, drawdowns, correlations, and market structure.

Historical analytics piece. Not investment advice. Not predictive modeling.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from db import get_engine

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
engine = get_engine()

## Database schema

In [ ]:
tables = pd.read_sql('''
    SELECT table_name,
           (SELECT COUNT(*) FROM information_schema.columns
            WHERE table_name = t.table_name AND table_schema = 'public') AS n_cols
    FROM information_schema.tables t
    WHERE table_schema = 'public'
    ORDER BY table_name
''', engine)
print(tables.to_string(index=False))

Table layers:

| Layer | Tables |
|---|---|
| Raw + adjusted prices | `prices_raw`, `prices`, `split_events`, `daily_returns` |
| Returns analysis | `monthly_returns`, `annual_returns`, `cumulative_returns` |
| Risk / ranking | `rolling_volatility`, `volatility_summary`, `performance_rankings`, `top_bottom_performers`, `yearly_leaders`, `risk_return_grid` |
| Drawdowns | `daily_drawdowns`, `drawdown_summary`, `broad_market_drops` |
| Correlations | `correlation_universe`, `pairwise_correlations`, `top_correlated_pairs`, `bottom_correlated_pairs` |
| Market structure | `ew_index_daily`, `ew_index_annual`, `monthly_volume_regimes` |

## ETL - split detection

The raw CSV does not split-adjust prices. A naive daily-return calculation will include false "crashes" on split dates. The ETL detects splits via a magnitude heuristic: single-day move >35% is treated as a split candidate. Adjustment factors are then applied to all historical prices for that symbol.

In [ ]:
splits = pd.read_sql('''
    SELECT symbol, date, prev_close, close, ROUND(adjustment_factor::numeric, 3) AS factor
    FROM split_events
    ORDER BY date
''', engine)
print(f"Split candidates detected: {len(splits)}\n")
print(splits.to_string(index=False))

Real corporate events from the period:

- **EBAY 2015-07-20**: PayPal spinoff (PYPL distributed to shareholders)
- **BAX 2015-07-01**: Baxalta spinoff
- **NI 2015-07-02**: Columbia Pipeline Group spinoff
- **DISCK/DISCA 2014-08-07**: 2-for-1 stock splits
- **LNT 2016-05-19/20**: 2-for-1 split
- **AMD 2016-04-22**: 52% earnings jump (not a split — false positive worth flagging)
- **VRTX 2014-06-24**: 40% jump on FDA approval of cystic fibrosis drug
- **NWL 2017-09-15**: large jump on news

The heuristic catches both genuine corporate actions and large news-driven moves. In a production pipeline you'd use a corporate-actions feed.

## Returns over the full period

In [ ]:
top = pd.read_sql('''
    SELECT symbol,
           ROUND(cumulative_return::numeric * 100, 1) AS cum_return_pct,
           ROUND(annualized_return::numeric * 100, 1) AS annualized_pct
    FROM top_bottom_performers WHERE bucket = 'TOP' AND rank_best <= 15
    ORDER BY rank_best
''', engine)
print("TOP 15 PERFORMERS")
print(top.to_string(index=False))

In [ ]:
bot = pd.read_sql('''
    SELECT symbol,
           ROUND(cumulative_return::numeric * 100, 1) AS cum_return_pct,
           ROUND(annualized_return::numeric * 100, 1) AS annualized_pct
    FROM top_bottom_performers WHERE bucket = 'BOTTOM' AND rank_best <= 15
    ORDER BY rank_best
''', engine)
print("BOTTOM 15 PERFORMERS")
print(bot.to_string(index=False))

NVDA returned 1,120% - the early years of the GPU/AI rally. Top of the list is dominated by semis (NVDA, AVGO, SWKS, LRCX), gaming (EA, ATVI), streaming/communications (NFLX, FB).

Bottom of the list is dominated by energy (CHK, RRC, NBL, NOV, MRO, APA, KMI, FCX). The 2014–2016 oil price collapse from $100 to $30 destroyed equity value across the sector.

### Yearly leaders

In [ ]:
leaders = pd.read_sql('''
    SELECT year, symbol, ROUND(annual_return::numeric * 100, 1) AS annual_pct
    FROM yearly_leaders ORDER BY year, year_rank
''', engine)
print(leaders.to_string(index=False))

## Volatility

In [ ]:
vol = pd.read_sql('''
    SELECT symbol,
           ROUND(vol_annualized::numeric * 100, 1) AS vol_pct,
           ROUND(mean_annualized::numeric * 100, 1) AS mean_return_pct,
           ROUND(return_to_vol_ratio::numeric, 2) AS sharpe_like
    FROM volatility_summary
    ORDER BY vol_annualized DESC LIMIT 10
''', engine)
print("MOST VOLATILE (annualized vol)")
print(vol.to_string(index=False))

Energy stocks (CHK, FCX, MRO, WMB, RRC) dominate the high-vol list - predictable for a sector going through a commodity collapse. AMD shows high vol with positive returns (turnaround story emerging in 2016–2017).

![Risk-return](figures/03_risk_return.png)

NVDA sits clearly above the cloud - exceptional returns without the highest volatility. Bottom-right (high vol + low return) is the worst quadrant; CHK is the clearest example.

## Drawdowns

In [ ]:
dd = pd.read_sql('''
    SELECT symbol, trough_date,
           ROUND(peak_price::numeric, 2) AS peak,
           ROUND(trough_price::numeric, 2) AS trough,
           ROUND(max_drawdown::numeric * 100, 1) AS max_dd_pct
    FROM drawdown_summary LIMIT 15
''', engine)
print("WORST DRAWDOWNS")
print(dd.to_string(index=False))

CHK fell from $31.30 to $1.59 - a 95% drawdown. Most of the worst drawdowns bottomed in early 2016, the peak of the oil panic. UAA/UA (Under Armour) bottomed in late 2017 after missing earnings.

### Worst market days

In [ ]:
drops = pd.read_sql('''
    SELECT date, n_down, n_down_2pct,
           ROUND(mean_return::numeric * 100, 2) AS mean_return_pct
    FROM broad_market_drops LIMIT 10
''', engine)
print(drops.to_string(index=False))

These dates align with known macro events:

- **2015-08-24**: "Black Monday" - China devalued the yuan; global markets crashed
- **2016-06-24**: Brexit vote
- **2015-09-01** / **2015-08-21**: continuation of the China growth scare
- **2016-01-13**: oil prices touched $30 for the first time
- **2016-09-09**: Fed rate-hike speculation

## Correlations

In [ ]:
corr = pd.read_sql('''
    SELECT symbol_a, symbol_b, ROUND(correlation::numeric, 3) AS corr
    FROM top_correlated_pairs LIMIT 10
''', engine)
print("MOST CORRELATED PAIRS")
print(corr.to_string(index=False))

These pairs are all same-industry:

- **AMAT/LRCX**: both semiconductor equipment
- **AVGO/SWKS**: both semiconductor
- **AET/ANTM/UNH/CI**: all health insurance
- **AVGO/TXN, LRCX/TXN**: semis again

Textbook sector-clustering pattern. Stocks in the same industry move together because they share common drivers.

## Market structure - equal-weighted index

In [ ]:
annual = pd.read_sql('SELECT * FROM ew_index_annual', engine)
annual["annual_return_pct"] = (annual["annual_return"] * 100).round(1)
print(annual[["year", "year_open", "year_close", "annual_return_pct"]].to_string(index=False))

The equal-weighted index gained ~55% over 4 years:

- **2014**: +16.5%
- **2015**: -1.1% (China scare wiped out the gains)
- **2016**: +15.3% (recovery)
- **2017**: +16.9% (broad-based rally)

![Equal-weighted index](figures/01_ew_index.png)

## Limitations

- **Split detection is heuristic.** Real splits and large news moves are both caught. In production, use a corporate-actions feed.
- **No sector data.** Sector analysis would add depth but requires external data.
- **No fundamentals.** Prices only.
- **2014–2017 was a bull market.** Findings don't generalize to bear markets.
- **No survivorship adjustment.** Companies delisted before 2014 aren't in the data.
- **Equal-weighted index is illustrative.** Not tradable, doesn't match the cap-weighted S&P 500.

In [ ]:
engine.dispose()